<a href="https://colab.research.google.com/github/yonatantussa/ST-GCN/blob/main/ST_GCN_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data Loader/Feeder

In [2]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import numpy as np
import scipy
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, cross_val_score


class load_data:
    def __init__(self, info,pose, cop,clin="motor_delay",thresh_spear=0.6,thresh_disp=45,exclude=["adjust","pose","error"], test_aim1=0.25, test_aim3=0.001, batch_size=32,seq_len=120,cop_mult=5):
        #Loading numpy files with COP and pose
        COP=np.load(cop)
        ##Pose data has already been filtered and zeroed to initial neck keypoint(along X/Y axis)
        Pose=np.load(pose)
        Pose_COP=self.format_data(COP,Pose,cop_mult)

        # Reading information summary excel file
        info = pd.read_excel(info)

        #Filtering out data with average spearman correlation less than threshold
        info_thresh1=info.loc[info["Avg Spear"]>=thresh_spear,:]

        #Filtering out data with disparity value less than threshold
        info_thresh2=info_thresh1.loc[info_thresh1["disparity"]<=thresh_disp,:]

        #Filtering out data with specified excusion tags
        info_filtered=info_thresh2.loc[~info_thresh2["exclude"].isin(exclude), :]

        #Isolataing rows with specific clinical test
        info_final=info_filtered.dropna(subset=[clin])

        #Separating data into aim 1 and 3
        aim1=info_final.loc[info_final["Aim"]==1,:]
        aim3=info_final.loc[info_final["Aim"]==3,:]

        #Implementing training and test splits based on aim
        train_set_aim1, test_set_aim1 = train_test_split(aim1,test_size=float(test_aim1))
        train_set_aim3, test_set_aim3 = train_test_split(aim3,test_size=float(test_aim3))

        #Stacking final test and training set
        train_info=pd.concat([train_set_aim1,train_set_aim3],ignore_index=True)
        test_info=pd.concat([test_set_aim1,test_set_aim3],ignore_index=True)

        print("Training Set Size: ", len(train_info),"Infants, ", train_info.window_round.sum()*120," Frames")
        print("Testing Set Size: ", len(test_info),"Infants, ", test_info.window_round.sum()*120," Frames")



        # turning data into dataset and splitting into 29, 120s windows
        self.training_set = PANDA_dateset(COP,Pose,Pose_COP,train_info,test=clin,window_size=seq_len)
        self.test_set = PANDA_dateset(COP,Pose,Pose_COP,test_info,test=clin,window_size=seq_len)

        # loading datasets into dataloader
        self.train_loader = DataLoader(self.training_set, batch_size=batch_size)
        self.test_loader = DataLoader(self.test_set, batch_size=batch_size)

        self.batch_size=batch_size
        # self.num_seq=num_seq
        self.seq_len=seq_len
        # self.scale=scale

        self.info=info

    def format_data(self,cop,pose,cop_multiplier):
        #Extracting dims of pose file
        n_subs,n_dim,n_joints,max_frames=np.shape(pose)
        #Empty matrix with room for extra(COP) keypoint
        pose_extra=np.zeros((n_subs,n_dim,n_joints+1,max_frames))

        #Filling pose keypoints
        pose_extra[:,:,0:n_joints,:]=pose

        #Extracting COP and remving offset from first 10 frames
        X_cop=cop[:,0,:]-np.mean(cop[:,0,0:10],axis=1, keepdims=True)
        Y_cop=cop[:,1,:]-np.mean(cop[:,1,0:10],axis=1, keepdims=True)
        Z_cop=np.zeros_like(X_cop)

        #Getting initial location from first 10 frames for neck, left hip and right hip
        offset_neck=np.mean(pose[:,:,1,0:10],axis=2, keepdims=True)
        offset_lhip=np.mean(pose[:,:,11,0:10],axis=2, keepdims=True)
        offset_rhip=np.mean(pose[:,:,8,0:10],axis=2, keepdims=True)

        #Final offset value as center point of intital location of neck, lhip and rhip
        offset=(offset_neck+offset_lhip+offset_rhip)/3

        #Filling in cop keypoints and adding offset
        pose_extra[:,0,-1,:]=X_cop*cop_multiplier+offset[:,0]
        pose_extra[:,1,-1,:]=Y_cop*cop_multiplier+offset[:,1]
        pose_extra[:,2,-1,:]=Z_cop+offset[:,2]

        return pose_extra
    # def gen_dataset(self,input_ids):
    #     id_bool=np.isin(self.ids, input_ids)
    #     dataset = PANDA_dateset(self.input[id_bool], self.output[id_bool], self.info[id_bool],self.model_cop[id_bool],window_size=self.seq_len)
    #     dataloader=DataLoader(dataset, batch_size=self.batch_size)

    #     return dataset,dataloader

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Class for PANDA daset as data loader object
class PANDA_dateset(Dataset):
    def __init__(self, cop, pose, pose_cop,info,test,window_size=120):
        self.info=info
        self.window_size = window_size
        self.dim_size=np.shape(pose)[1]
        # datset parameters
        self.tot_subjects = len(info)

        # data typer parameters
        self.num_kp = 18

        scale=self.get_scale(pose)
        clinical=info[test]

        # Reading cop and pose files in pytorch
        cop = torch.from_numpy(cop).float()
        pose = torch.from_numpy(pose).float()
        pose_cop = torch.from_numpy(pose_cop).float()


        #Empty
        Pose = []
        Cop = []
        Pose_Cop = []
        scale_batch=[]
        clin_outcome=[]

        # unrolling data for each subject into 29, 120-frame windows
        for i in range(len(info)):
          subject=info.iloc[i]

          num_win=subject.window_round
          tot_frames=int(num_win*window_size)
          idx=subject.indx
          # print(tot_frames)

          # updating pose
          pose_stack = pose[idx, :, :, : tot_frames]
          # updating cop
          cop_stack = cop[idx, :, : tot_frames]
          # updating Pose COP
          pose_cop_stack = pose_cop[i, :, : tot_frames]

          #Getting data unrolled and put in batches
          pose_unroll,cop_unroll,pose_cop_unroll,clin_unroll=self.batch_data(pose_stack,cop_stack,pose_cop_stack,clinical[i],num_seq=num_win)

          #Appending final answert to results
          Pose.append(pose_unroll)
          Cop.append(cop_unroll)
          Pose_Cop.append(pose_cop_unroll)
          scale_batch.append(scale[i]*torch.ones(num_win))
          clin_outcome.append(clin_unroll.bool())

        #Save rolled out data (unformatted timesieres)
        self.pose_roll = pose
        self.cop_roll = cop
        self.model_cop_roll = pose_cop


        #Save batched data
        self.pose = torch.cat(Pose, dim=0)
        self.cop = torch.cat(Cop, dim=0)
        self.pose_cop = torch.cat(Pose_Cop, dim=0)
        self.clin = torch.cat(clin_outcome, dim=0)
        self.scale_batch=torch.cat(scale_batch, dim=0)

        # print(self.pose.shape,self.cop.shape,self.model_cop.shape)


    def __len__(self):
        return int(self.num_seq.sum())

    def __getitem__(self, idx):
        item = {"pose": self.pose[idx, :, :, :], "cop": self.cop[idx, :, :], "pose_cop": self.pose_cop[idx, :, :], "clinical":self.clin[idx],"scale": self.scale_batch[idx]}
        return item

    def get_scale(self,pose):
        neck = pose[:, :, 1, 0:10]
        mid_hip = (pose[:, :, 8, 0:10] + pose[:, :, 11, 0:10]) / 2

        lengths = np.linalg.norm(neck - mid_hip, axis=1)
        scale = np.mean(lengths, axis=1)

        return torch.from_numpy(scale).float()


    def batch_data(self,pose,cop,pose_cop,clin,num_seq):
        num_windows=num_seq
        window_size=self.window_size

        clin_unroll = torch.ones((num_windows))*clin

        pose_unroll = torch.zeros((num_windows, self.dim_size, self.num_kp, self.window_size))
        cop_unroll = torch.zeros((num_windows, 2, window_size))
        pose_cop_unroll = torch.zeros((num_windows, self.dim_size, self.num_kp+1, self.window_size))


        # Pose/cop_pose[n_dim,n_joints,frames]
        for j in range(num_windows):
          # updating pose
          pose_unroll[j, :, :, :] = pose[:, :, (j * window_size) : (j * window_size) + window_size]
          # updating cop
          cop_unroll[j, :, :] = cop[:, (j * window_size) : (j * window_size) + window_size]
          # updating model cop
          pose_cop_unroll[j, :, :,:] = pose_cop[:, :,(j * window_size) : (j * window_size) + window_size]

        return pose_unroll,cop_unroll,pose_cop_unroll,clin_unroll




# Neural Nets

Graph Representation

In [5]:
import numpy as np

class Graph():
    """ The Graph to model the skeletons extracted by the openpose

    Args:
        strategy (string): must be one of the follow candidates
        - uniform: Uniform Labeling
        - distance: Distance Partitioning
        - spatial: Spatial Configuration
        For more information, please refer to the section 'Partition Strategies'
            in our paper (https://arxiv.org/abs/1801.07455).

        layout (string): must be one of the follow candidates
        - openpose: Is consists of 18 joints. For more information, please
            refer to https://github.com/CMU-Perceptual-Computing-Lab/openpose#output
        - ntu-rgb+d: Is consists of 25 joints. For more information, please
            refer to https://github.com/shahroudy/NTURGB-D

        max_hop (int): the maximal distance between two connected nodes
        dilation (int): controls the spacing between the kernel points

    """

    def __init__(self,
                 layout='openpose',
                 strategy='spatial',
                 max_hop=1,
                 dilation=1):
        self.max_hop = max_hop
        self.dilation = dilation

        self.get_edge(layout)
        self.hop_dis = get_hop_distance(
            self.num_node, self.edge, max_hop=max_hop)
        self.get_adjacency(strategy)

    def __str__(self):
        return self.A

    def get_edge(self, layout):
        if layout == 'pose':
            self.num_node = 18
            self_link = [(i, i) for i in range(self.num_node)]
            neighbor_link = [
                [0, 1],
                [1, 2],
                [2, 3],
                [3, 4],
                [1, 5],
                [5, 6],
                [6, 7],
                [1, 8],
                [8, 9],
                [9, 10],
                [1, 11],
                [11, 12],
                [12, 13],
                [0, 14],
                [14, 16],
                [0, 15],
                [15, 17],
            ]
            self.edge = self_link + neighbor_link
            self.center = 1
        elif layout == 'cop_pose':
            self.num_node = 19
            self_link = [(i, i) for i in range(self.num_node)]
            neighbor_link = [
                [0, 1],
                [1, 2],
                [2, 3],
                [3, 4],
                [1, 5],
                [5, 6],
                [6, 7],
                [1, 8],
                [8, 9],
                [9, 10],
                [1, 11],
                [11, 12],
                [12, 13],
                [0, 14],
                [14, 16],
                [0, 15],
                [15, 17],
                [1,19]
            ]
            self.edge = self_link + neighbor_link
            self.center = 1
        else:
            raise ValueError("Do Not Exist This Layout.")

    def get_adjacency(self, strategy):
        valid_hop = range(0, self.max_hop + 1, self.dilation)
        adjacency = np.zeros((self.num_node, self.num_node))
        for hop in valid_hop:
            adjacency[self.hop_dis == hop] = 1
        normalize_adjacency = normalize_digraph(adjacency)

        if strategy == 'uniform':
            A = np.zeros((1, self.num_node, self.num_node))
            A[0] = normalize_adjacency
            self.A = A
        elif strategy == 'distance':
            A = np.zeros((len(valid_hop), self.num_node, self.num_node))
            for i, hop in enumerate(valid_hop):
                A[i][self.hop_dis == hop] = normalize_adjacency[self.hop_dis ==
                                                                hop]
            self.A = A
        elif strategy == 'spatial':
            A = []
            for hop in valid_hop:
                a_root = np.zeros((self.num_node, self.num_node))
                a_close = np.zeros((self.num_node, self.num_node))
                a_further = np.zeros((self.num_node, self.num_node))
                for i in range(self.num_node):
                    for j in range(self.num_node):
                        if self.hop_dis[j, i] == hop:
                            if self.hop_dis[j, self.center] == self.hop_dis[
                                    i, self.center]:
                                a_root[j, i] = normalize_adjacency[j, i]
                            elif self.hop_dis[j, self.
                                              center] > self.hop_dis[i, self.
                                                                     center]:
                                a_close[j, i] = normalize_adjacency[j, i]
                            else:
                                a_further[j, i] = normalize_adjacency[j, i]
                if hop == 0:
                    A.append(a_root)
                else:
                    A.append(a_root + a_close)
                    A.append(a_further)
            A = np.stack(A)
            self.A = A
        else:
            raise ValueError("Do Not Exist This Strategy")


def get_hop_distance(num_node, edge, max_hop=1):
    A = np.zeros((num_node, num_node))
    for i, j in edge:
        A[j, i] = 1
        A[i, j] = 1

    # compute hop steps
    hop_dis = np.zeros((num_node, num_node)) + np.inf
    transfer_mat = [np.linalg.matrix_power(A, d) for d in range(max_hop + 1)]
    arrive_mat = (np.stack(transfer_mat) > 0)
    for d in range(max_hop, -1, -1):
        hop_dis[arrive_mat[d]] = d
    return hop_dis


def normalize_digraph(A):
    Dl = np.sum(A, 0)
    num_node = A.shape[0]
    Dn = np.zeros((num_node, num_node))
    for i in range(num_node):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i]**(-1)
    AD = np.dot(A, Dn)
    return AD


def normalize_undigraph(A):
    Dl = np.sum(A, 0)
    num_node = A.shape[0]
    Dn = np.zeros((num_node, num_node))
    for i in range(num_node):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i]**(-0.5)
    DAD = np.dot(np.dot(Dn, A), Dn)
    return DAD

TGCN

In [6]:
# The based unit of graph convolutional networks.

import torch
import torch.nn as nn

class ConvTemporalGraphical(nn.Module):

    r"""The basic module for applying a graph convolution.

    Args:
        in_channels (int): Number of channels in the input sequence data
        out_channels (int): Number of channels produced by the convolution
        kernel_size (int): Size of the graph convolving kernel
        t_kernel_size (int): Size of the temporal convolving kernel
        t_stride (int, optional): Stride of the temporal convolution. Default: 1
        t_padding (int, optional): Temporal zero-padding added to both sides of
            the input. Default: 0
        t_dilation (int, optional): Spacing between temporal kernel elements.
            Default: 1
        bias (bool, optional): If ``True``, adds a learnable bias to the output.
            Default: ``True``

    Shape:
        - Input[0]: Input graph sequence in :math:`(N, in_channels, T_{in}, V)` format
        - Input[1]: Input graph adjacency matrix in :math:`(K, V, V)` format
        - Output[0]: Outpu graph sequence in :math:`(N, out_channels, T_{out}, V)` format
        - Output[1]: Graph adjacency matrix for output data in :math:`(K, V, V)` format

        where
            :math:`N` is a batch size,
            :math:`K` is the spatial kernel size, as :math:`K == kernel_size[1]`,
            :math:`T_{in}/T_{out}` is a length of input/output sequence,
            :math:`V` is the number of graph nodes.
    """

    def __init__(self,
                 in_channels,
                 out_channels,
                 kernel_size,
                 t_kernel_size=1,
                 t_stride=1,
                 t_padding=0,
                 t_dilation=1,
                 bias=True):
        super().__init__()

        self.kernel_size = kernel_size
        self.conv = nn.Conv2d(
            in_channels,
            out_channels * kernel_size,
            kernel_size=(t_kernel_size, 1),
            padding=(t_padding, 0),
            stride=(t_stride, 1),
            dilation=(t_dilation, 1),
            bias=bias)

    def forward(self, x, A):
        assert A.size(0) == self.kernel_size

        x = self.conv(x)

        n, kc, t, v = x.size()
        x = x.view(n, self.kernel_size, kc//self.kernel_size, t, v)
        x = torch.einsum('nkctv,kvw->nctw', (x, A))

        return x.contiguous(), A


ST-GCN Unit

In [7]:
# The based unit of graph convolutional networks.

import torch
import torch.nn as nn

# import ConvTemporalGraphical
# import Bi_LSTM
# import Graph

class st_gcn(nn.Module):
    r"""Applies a spatial temporal graph convolution over an input graph sequence.

    Args:
        in_channels (int): Number of channels in the input sequence data
        out_channels (int): Number of channels produced by the convolution
        kernel_size (tuple): Size of the temporal convolving kernel and graph convolving kernel
        stride (int, optional): Stride of the temporal convolution. Default: 1
        dropout (int, optional): Dropout rate of the final output. Default: 0
        residual (bool, optional): If ``True``, applies a residual mechanism. Default: ``True``

    Shape:
        - Input[0]: Input graph sequence in :math:`(N, in_channels, T_{in}, V)` format
        - Input[1]: Input graph adjacency matrix in :math:`(K, V, V)` format
        - Output[0]: Outpu graph sequence in :math:`(N, out_channels, T_{out}, V)` format
        - Output[1]: Graph adjacency matrix for output data in :math:`(K, V, V)` format

        where
            :math:`N` is a batch size,
            :math:`K` is the spatial kernel size, as :math:`K == kernel_size[1]`,
            :math:`T_{in}/T_{out}` is a length of input/output sequence,
            :math:`V` is the number of graph nodes.

    """

    def __init__(self, in_channels, out_channels, kernel_size, stride=1, dropout=0.5, residual=True):
        super().__init__()

        assert len(kernel_size) == 2
        assert kernel_size[0] % 2 == 1
        padding = ((kernel_size[0] - 1) // 2, 0)

        self.gcn = ConvTemporalGraphical(in_channels, out_channels, kernel_size[1])

        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=False),
            nn.Dropout(dropout, inplace=False),
            nn.Conv2d(
                out_channels,
                out_channels,
                (kernel_size[0], 1),
                (stride, 1),
                padding,
            ),
            nn.BatchNorm2d(out_channels),
        )

        if not residual:
            self.residual = lambda x: 0

        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x

        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )

        self.relu = nn.ReLU(inplace=False)

    def forward(self, x, A):

        res = self.residual(x)
        x, A = self.gcn(x, A)
        x = self.tcn(x) + res

        return self.relu(x), A

Full Model

In [8]:
import torch
import torch.nn as nn

# import ConvTemporalGraphical
# import Bi_LSTM
# import Graph

class Model(nn.Module):
    r"""Spatial temporal graph convolutional networks.

    Args:
        in_channels (int): Number of channels in the input data
        num_class (int): Number of classes for the classification task
        graph_args (dict): The arguments for building the graph
        edge_importance_weighting (bool): If ``True``, adds a learnable
            importance weighting to the edges of the graph
        **kwargs (optional): Other parameters for graph convolution units

    Shape:
        - Input: :math:`(N, in_channels, T_{in}, V_{in}, M_{in})`
        - Output: :math:`(N, num_class)` where
            :math:`N` is a batch size,
            :math:`T_{in}` is a length of input sequence,
            :math:`V_{in}` is the number of graph nodes,
            :math:`M_{in}` is the number of instance in a frame.
    """

    def __init__(self, in_channels, num_class, graph_args,
                 edge_importance_weighting, **kwargs):
        super().__init__()

        # load graph
        self.graph = Graph(**graph_args)
        A = torch.tensor(self.graph.A, dtype=torch.float32, requires_grad=False)
        self.register_buffer('A', A)

        # build networks
        spatial_kernel_size = A.size(0)
        temporal_kernel_size = 9
        kernel_size = (temporal_kernel_size, spatial_kernel_size)
        self.data_bn = nn.BatchNorm1d(in_channels * A.size(1))
        kwargs0 = {k: v for k, v in kwargs.items() if k != 'dropout'}
        self.st_gcn_networks = nn.ModuleList((
            st_gcn(in_channels, 64, kernel_size, 1, residual=False, **kwargs0),
            st_gcn(64, 64, kernel_size, 1, **kwargs),
            st_gcn(64, 64, kernel_size, 1, **kwargs),
            st_gcn(64, 64, kernel_size, 1, **kwargs),
            st_gcn(64, 128, kernel_size, 2, **kwargs),
            st_gcn(128, 128, kernel_size, 1, **kwargs),
            st_gcn(128, 128, kernel_size, 1, **kwargs),
            st_gcn(128, 256, kernel_size, 2, **kwargs),
            st_gcn(256, 256, kernel_size, 1, **kwargs),
            st_gcn(256, 256, kernel_size, 1, **kwargs),
        ))

        # initialize parameters for edge importance weighting
        if edge_importance_weighting:
            self.edge_importance = nn.ParameterList([
                nn.Parameter(torch.ones(self.A.size()))
                for i in self.st_gcn_networks
            ])
        else:
            self.edge_importance = [1] * len(self.st_gcn_networks)

        # fcn for prediction
        self.fcn = nn.Conv2d(256, num_class, kernel_size=1)

    def forward(self, x):

        # data normalization
        N, C, T, V, M = x.size()
        x = x.permute(0, 4, 3, 1, 2).contiguous()
        x = x.view(N * M, V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T)
        x = x.permute(0, 1, 3, 4, 2).contiguous()
        x = x.view(N * M, C, T, V)

        # forwad
        for gcn, importance in zip(self.st_gcn_networks, self.edge_importance):
            x, _ = gcn(x, self.A * importance)

        # global pooling
        x = F.avg_pool2d(x, x.size()[2:])
        x = x.view(N, M, -1, 1, 1).mean(dim=1)

        # prediction
        x = self.fcn(x)
        x = x.view(x.size(0), -1)

        return x

    def extract_feature(self, x):

        # data normalization
        N, C, T, V, M = x.size()
        x = x.permute(0, 4, 3, 1, 2).contiguous()
        x = x.view(N * M, V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T)
        x = x.permute(0, 1, 3, 4, 2).contiguous()
        x = x.view(N * M, C, T, V)

        # forwad
        for gcn, importance in zip(self.st_gcn_networks, self.edge_importance):
            x, _ = gcn(x, self.A * importance)

        _, c, t, v = x.size()
        feature = x.view(N, M, c, t, v).permute(0, 2, 3, 4, 1)

        # prediction
        x = self.fcn(x)
        output = x.view(N, M, -1, t, v).permute(0, 2, 3, 4, 1)

        return output, feature

## Training Model

In [9]:
%pip install tqdm



In [10]:
import os

# changed path to MyDrive due to issues with Shared
posefile = "/content/drive/MyDrive/content/pose_dat.npy"
copfile = "/content/drive/MyDrive/content/cop_dat.npy"
infofile = "/content/drive/MyDrive/content/Final Data.xlsx"

# Verification check to see if files exist in Google Drive
for file_path in [posefile, copfile, infofile]:
    if not os.path.exists(file_path):
        print(f"Warning: File not found at {file_path}. Please make sure you have uploaded it to the session or that it is located in the correct directory.")

loader=load_data(infofile,posefile,copfile,)

Training Set Size:  261 Infants,  876720  Frames
Testing Set Size:  19 Infants,  61560  Frames


In [11]:
import random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix

# Configure random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Training settings
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.3

# Set checkpoint path to Google Drive to persist weights across sessions
CHECKPOINT_PATH = Path('/content/drive/MyDrive/content/st_gcn_final.pt')

def training_tensor_loader(dataset, batch_size, shuffle):
    """Use tensors already created by the collaborator's dataset, unchanged."""
    poses = dataset.pose.float()  # N,C,V,T
    labels = dataset.clin.reshape(-1).long()
    if poses.ndim != 4 or poses.shape[2] != 18:
        raise ValueError('Expected pose windows with shape (N,C,18,T).')
    if len(poses) != len(labels) or len(labels) == 0:
        raise ValueError('Pose/label lengths differ or split is empty.')
    if not torch.isfinite(poses).all():
        raise ValueError('Pose contains nonfinite coordinates.')
    if not set(dataset.info['motor_delay'].dropna().unique()).issubset({0,1}):
        raise ValueError('This training section requires binary motor_delay labels 0/1.')
    return DataLoader(TensorDataset(poses, labels), batch_size=batch_size, shuffle=shuffle)

train_batches = training_tensor_loader(loader.training_set, loader.batch_size, shuffle=True)
test_batches = training_tensor_loader(loader.test_set, loader.batch_size, shuffle=False)

model = Model(
    in_channels=loader.training_set.pose.shape[1],
    num_class=2,
    graph_args={'layout':'pose', 'strategy':'spatial'},
    edge_importance_weighting=True,
    dropout=DROPOUT,
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print('Training on:', device)

Training on: cuda


### 3. Model Training
This block runs the forward and backward passes over our ST-GCN model, computes the binary cross-entropy loss, clips gradients to prevent instability, and saves the final model checkpoint.

In [12]:
def run_training_epoch(model, batches, optimizer):
    model.train()
    loss_sum, correct, count = 0.0, 0, 0
    for poses, labels in batches:
        # Original windows: N,C,V,T -> model input: N,C,T,V,M; M=1 infant.
        x = poses.permute(0,1,3,2).contiguous().unsqueeze(-1).to(device)
        y = labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)  # raw logits; CrossEntropyLoss applies log-softmax
        loss = criterion(logits, y)
        if not torch.isfinite(loss):
            raise RuntimeError('Nonfinite loss; inspect input coordinates.')
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        loss_sum += loss.item()*len(y)
        correct += (logits.argmax(dim=1)==y).sum().item()
        count += len(y)
    return loss_sum/count, correct/count

history = []
start_epoch = 1

# Resuming training if checkpoint exists in Drive
if CHECKPOINT_PATH.exists():
    print(f"Checkpoint found at {CHECKPOINT_PATH}. Loading progress...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint.get('epochs', 0) + 1
    history = checkpoint.get('history', [])
    print(f"Resuming training from epoch {start_epoch}")

for epoch in range(start_epoch, NUM_EPOCHS+1):
    train_loss, train_accuracy = run_training_epoch(model, train_batches, optimizer)
    history.append({'epoch':epoch,'train_loss':train_loss,'train_accuracy':train_accuracy})
    print(f'Epoch {epoch:03d}/{NUM_EPOCHS}: loss={train_loss:.4f}, accuracy={train_accuracy:.4f}')

    # Periodically save backup after each epoch directly to Google Drive
    torch.save({
        'model_state_dict':model.state_dict(),
        'optimizer_state_dict':optimizer.state_dict(),
        'epochs':epoch,
        'in_channels':loader.training_set.pose.shape[1],
        'num_class':2,
        'graph_args':{'layout':'pose','strategy':'spatial'},
        'edge_importance_weighting':True,
        'dropout':DROPOUT,
        'history':history,
        'seed':SEED,
    }, CHECKPOINT_PATH)

print('Saved checkpoint to:', CHECKPOINT_PATH)

Checkpoint found at /content/drive/MyDrive/content/st_gcn_final.pt. Loading progress...
Resuming training from epoch 51
Saved checkpoint to: /content/drive/MyDrive/content/st_gcn_final.pt


In [13]:
@torch.no_grad()
def evaluate(model, batches):
    model.eval()
    targets, probabilities = [], []
    loss_sum, count = 0.0, 0
    for poses, labels in batches:
        x = poses.permute(0,1,3,2).contiguous().unsqueeze(-1).to(device)
        y = labels.to(device)
        logits = model(x)
        loss_sum += criterion(logits,y).item()*len(y)
        count += len(y)
        targets.extend(y.cpu().tolist())
        probabilities.extend(logits.softmax(dim=1)[:,1].cpu().tolist())
    targets = np.asarray(targets)
    probabilities = np.asarray(probabilities)
    predictions = (probabilities >= 0.5).astype(int)
    return {
        'loss':loss_sum/count,
        'accuracy':accuracy_score(targets,predictions),
        'balanced_accuracy':balanced_accuracy_score(targets,predictions),
        'auroc':roc_auc_score(targets,probabilities) if len(np.unique(targets))==2 else float('nan'),
        'confusion_matrix':confusion_matrix(targets,predictions,labels=[0,1]),
    }

# Window-level metrics; correlated windows do not count as independent infants.
test_metrics = evaluate(model, test_batches)
for name,value in test_metrics.items():
    print(name, ':', value)


loss : 0.42627194126252504
accuracy : 0.9395711500974658
balanced_accuracy : 0.8590954630027193
auroc : 0.9500262392061447
confusion_matrix : [[408   3]
 [ 28  74]]
